# Minimal cross-domain query — Netherlands 2024

Pull **CO₂** from the atmosphere store, **monthly NEE** from the ecosystem
store, and **fCO₂** from the ocean store for stations / cruises inside a
Netherlands bounding box, restricted to 2024 — three domains, one pattern.

- **Region**: lat 50.7–53.6, lon 3.3–7.3
- **Time**: 2024-01-01 → 2024-12-31

Everything reads **directly from the public ICOS zarr proxy** at
`https://zarr.icos-cp.eu` via `xr.open_zarr(URL, consolidated=True)`:
one `.zmetadata` fetch, then only the chunks the query touches.

Each store has a *combined view* where the per-station / per-cruise data
shares one indexed dimension, so spatial + temporal selection is a single
xarray expression:

| Store | Combined group | Index dim | Spatial coords |
|---|---|---|---|
| `icos-obspack.zarr` | `co2`, `ch4`, `n2o`, `co` | `station` | `latitude(station)`, `longitude(station)`, `sampling_height(station)` |
| `icos-fluxnet.zarr` | `_combined/fluxnet_dd` … `_yy` | `station` | `latitude(station)`, `longitude(station)` |
| `icos-socat.zarr`   | `_obs` (flat obs table) | `obs` | `latitude(obs)`, `longitude(obs)`, `time(obs)`, `deployment(obs)` |

*Advanced example — notebook 12 of the [sample series](README.md). Builds on
notebooks 02/04/05 (one store each) and 08 (combined views). No account
needed. The same selections are also one `/query` URL each — see notebook 03
— but here we stay in pure xarray.*

In [1]:
import xarray as xr

BASE_URL = "https://zarr.icos-cp.eu"   # public ICOS zarr proxy

LAT_MIN, LAT_MAX = 50.7, 53.6
LON_MIN, LON_MAX = 3.3, 7.3
T0, T1 = "2024-01-01", "2024-12-31"

## Obspack — CO2

In [2]:
# set_coords promotes the per-station metadata columns to coordinates, so they
# travel with any DataArray extracted from the panel (they are stored as plain
# data variables).
ds = (xr.open_zarr(f"{BASE_URL}/icos-obspack.zarr/co2", consolidated=True)
      .set_coords(["latitude", "longitude", "sampling_height"]))

# Materialise the boolean indexer first — newer xarray rejects a boolean *dask*
# array in .where(..., drop=True); the 1-D station coords are tiny so it's cheap.
mask = (
    (ds.latitude >= LAT_MIN) & (ds.latitude <= LAT_MAX) &
    (ds.longitude >= LON_MIN) & (ds.longitude <= LON_MAX)
).compute()
co2_nl = ds["co2"].where(mask, drop=True).sel(time_co2=slice(T0, T1))

# Per-station summary — coords travel with the DataArray
for sid in co2_nl.station.values:
    da = co2_nl.sel(station=sid)
    print(f"{sid:8s} lat={float(co2_nl.latitude.sel(station=sid)):.3f} "
          f"lon={float(co2_nl.longitude.sel(station=sid)):.3f}  "
          f"height={float(co2_nl.sampling_height.sel(station=sid)):>4.0f} m  "
          f"n={int(da.count())}  "
          f"mean={float(da.mean()):.2f} ppm")

CBW127   lat=51.970 lon=4.926  height= 127 m  n=8549  mean=432.69 ppm


CBW207   lat=51.970 lon=4.926  height= 207 m  n=8739  mean=430.80 ppm


CBW27    lat=51.970 lon=4.926  height=  27 m  n=8526  mean=439.14 ppm


CBW67    lat=51.970 lon=4.926  height=  67 m  n=8541  mean=435.16 ppm


JUE120   lat=50.910 lon=6.410  height= 120 m  n=8443  mean=434.00 ppm


JUE50    lat=50.910 lon=6.410  height=  50 m  n=7996  mean=436.75 ppm


JUE80    lat=50.910 lon=6.410  height=  80 m  n=7992  mean=435.28 ppm


LUT60    lat=53.404 lon=6.353  height=  60 m  n=8332  mean=431.79 ppm


## FLUXNET — monthly NEE

Selects the VUT/REF NEE variant from each site's `fluxnet_mm` sub-group.

In [3]:
ds = (xr.open_zarr(f"{BASE_URL}/icos-fluxnet.zarr/_combined/fluxnet_mm",
                   consolidated=True)
      .set_coords(["latitude", "longitude"]))

mask = (
    (ds.latitude >= LAT_MIN) & (ds.latitude <= LAT_MAX) &
    (ds.longitude >= LON_MIN) & (ds.longitude <= LON_MAX)
).compute()
nee_nl = (
    ds["NEE"]
      .sel(ustar_threshold="VUT", nee_variant="REF")
      .where(ds["NEE_QC"].sel(ustar_threshold="VUT", nee_variant="REF") > 0.3)
      .where(mask, drop=True)
      .sel(time=slice(T0, T1))
)

units = ds["NEE"].attrs.get("units", "")
for sid in nee_nl.station.values:
    da = nee_nl.sel(station=sid)
    print(f"{sid:8s} lat={float(nee_nl.latitude.sel(station=sid)):.3f} "
          f"lon={float(nee_nl.longitude.sel(station=sid)):.3f}  "
          f"n={int(da.count())}  "
          f"mean={float(da.mean()):.3f} {units}")

BE-Bra   lat=51.308 lon=4.520  n=12  mean=-0.795 umol m-2 s-1
BE-Maa   lat=50.980 lon=5.632  n=12  mean=-0.062 umol m-2 s-1
DE-RuS   lat=50.866 lon=6.447  n=12  mean=-0.563 umol m-2 s-1
NL-Loo   lat=52.166 lon=5.744  n=12  mean=-0.407 umol m-2 s-1


## SOCAT — fCO₂

Same pattern as the other two stores, but `_obs` is a flat
`(obs,)` table: `time`, `lon`, `lat`, `deployment` are all 1-D
coords along `obs`. Bounding-box + time + QC selection is a single
`.where(...)` — no `.sel(time=slice(...))` because `time` is a
non-monotonic non-dim coord here (timestamps can collide across
cruises).


In [4]:
import numpy as np

ds = xr.open_zarr(f"{BASE_URL}/icos-socat.zarr/_obs", consolidated=True)

T0_64 = np.datetime64(T0)
T1_64 = np.datetime64(T1) + np.timedelta64(1, "D")   # T1 inclusive

mask = (
    (ds.latitude >= LAT_MIN) & (ds.latitude <= LAT_MAX) &
    (ds.longitude >= LON_MIN) & (ds.longitude <= LON_MAX) &
    (ds.time >= T0_64) & (ds.time < T1_64) &
    (ds.fCO2_QC == 2)                                # WOCE: keep flag 2 (good)
).compute()
sel = ds.where(mask, drop=True)

units = ds["fCO2"].attrs.get("units", "µatm")
v = sel["fCO2"].values
v = v[np.isfinite(v)]
print(f"SOCAT fCO2 — {sel.sizes.get('obs', 0)} matching observations  "
      f"→  {v.size} finite QC-passing samples")

if v.size:
    print(f"\npooled mean fCO2 = {v.mean():.1f} {units}  "
          f"(range {v.min():.1f}–{v.max():.1f})")
    dep_idx = sel["deployment"].values
    for i in np.unique(dep_idx):
        m = dep_idx == i
        vi = sel["fCO2"].values[m]
        vi = vi[np.isfinite(vi)]
        if not vi.size:
            continue
        cruise_id = str(ds.cruise.values[int(i)])
        station   = str(ds.station_id.values[int(i)])
        print(f"  {cruise_id:14s}  {station:30s}  n={vi.size:>5d}  "
              f"mean={vi.mean():.1f} {units}")

SOCAT fCO2 — 3152 matching observations  →  3152 finite QC-passing samples

pooled mean fCO2 = 421.4 uatm  (range 252.6–799.3)
  11SS20240501    BE-SOOP-Simon Stevin            n=   28  mean=288.3 uatm
  11SS20240601    BE-SOOP-Simon Stevin            n= 2382  mean=384.3 uatm


  11SS20240801    BE-SOOP-Simon Stevin            n=  276  mean=648.5 uatm
  11SS20241001    BE-SOOP-Simon Stevin            n=  128  mean=524.7 uatm
  11SS20241101    BE-SOOP-Simon Stevin            n=  268  mean=464.8 uatm
  11SS20241201    BE-SOOP-Simon Stevin            n=   22  mean=545.6 uatm


  26T320231110    FR-SOOP-France-Brazil           n=    1  mean=252.6 uatm
  26T320240719    FR-SOOP-France-Brazil           n=   47  mean=459.0 uatm


## Data passports — one per store, on demand

The proxy recorded every chunk the three selections touched, one session per
store. Asking for the passport closes each session and returns the RO-Crate
record of exactly what this analysis read — the citable footprint of the
Netherlands-2024 extraction.

In [5]:
import json, requests

for store in ("icos-obspack.zarr", "icos-fluxnet.zarr", "icos-socat.zarr"):
    r = requests.get(f"{BASE_URL}/{store}/session/passport").json()
    rec = r["passport"]["@graph"][1]
    fn = f"nl2024_{store.split('.')[0]}_passport.json"
    with open(fn, "w") as f:
        json.dump(r["passport"], f, indent=2)
    print(f"{store:22s} groups={rec['accessedGroups']}  "
          f"bytes={rec['totalBytesServed']:>10,}  → {fn}")

icos-obspack.zarr      groups=['CBW207', 'co2']  bytes=97,298,743  → nl2024_icos-obspack_passport.json
icos-fluxnet.zarr      groups=['_combined/fluxnet_mm']  bytes= 1,432,542  → nl2024_icos-fluxnet_passport.json
icos-socat.zarr        groups=['119920180214', '119920220401', '119920220701', '119920221001', '119920230501', '119920230701', '119920240601', '119920240902', '119920241202', '119920250401', '119920250601', '11SS20201023', '11SS20210104', '11SS20210201', '11SS20210301', '11SS20210401', '11SS20210501', '11SS20210601', '11SS20210701', '11SS20210801', '11SS20211001', '11SS20211101', '11SS20211206', '11SS20220101', '11SS20220201', '11SS20220301', '11SS20220401', '11SS20220501', '11SS20220601', '11SS20220701', '11SS20220801', '11SS20220901', '11SS20221001', '11SS20221101', '11SS20221121', '11SS20221201', '11SS20230101', '11SS20230201', '11SS20230301', '11SS20230401', '11SS20230501', '11SS20230601', '11SS20230701', '11SS20230801', '11SS20230901', '11SS20231001', '11SS20231101', '11S